In [1]:
import pandas as pd
import numpy as np

train = pd.read_parquet("../data/processed/train.parquet")
validation = pd.read_parquet("../data/processed/validation.parquet")

print("Train:", train.shape)
print("Validation:", validation.shape)

Train: (1928949, 5)
Validation: (413345, 5)


In [2]:
item_popularity = (
    train.groupby("item_id")
    .size()
    .sort_values(ascending=False)
)

print(item_popularity.head(20))

item_id
5411      1887
309778    1598
461686    1548
370653    1483
298009    1347
257040    1293
335975    1233
369447    1191
7943      1143
96924     1097
111530    1084
354233     974
37029      965
48030      948
441668     932
445351     900
234255     888
29196      840
312728     837
315543     831
dtype: int64


In [3]:
top_products = item_popularity.head(20)

print(top_products)

item_id
5411      1887
309778    1598
461686    1548
370653    1483
298009    1347
257040    1293
335975    1233
369447    1191
7943      1143
96924     1097
111530    1084
354233     974
37029      965
48030      948
441668     932
445351     900
234255     888
29196      840
312728     837
315543     831
dtype: int64


In [4]:
popular_items = item_popularity.index.tolist()

print("Number of candidate items:", len(popular_items))
print("Top 10:", popular_items[:10])

Number of candidate items: 200974
Top 10: [5411, 309778, 461686, 370653, 298009, 257040, 335975, 369447, 7943, 96924]


In [5]:
user_history = (
    train.groupby("user_id")["item_id"]
    .apply(set)
    .to_dict()
)

In [6]:
ground_truth = (
    validation.groupby("user_id")["item_id"]
    .apply(set)
    .to_dict()
)

In [7]:
def recommend_popular(user_id, k=10):
    seen_items = user_history.get(user_id, set())

    recommendations = []

    for item_id in popular_items:
        if item_id not in seen_items:
            recommendations.append(item_id)

        if len(recommendations) == k:
            break

    return recommendations

In [8]:
sample_user = next(iter(ground_truth))

print("User:", sample_user)
print("Recommendations:", recommend_popular(sample_user, 10))
print("Actual future items:", ground_truth[sample_user])

User: 1
Recommendations: [5411, 309778, 461686, 370653, 298009, 257040, 335975, 369447, 7943, 96924]
Actual future items: {72028}


In [9]:
def precision_at_k(recommended, relevant, k):
    recommended = recommended[:k]

    if k == 0:
        return 0.0

    hits = sum(item in relevant for item in recommended)

    return hits / k

In [10]:
def recall_at_k(recommended, relevant, k):
    recommended = recommended[:k]

    if len(relevant) == 0:
        return 0.0

    hits = sum(item in relevant for item in recommended)

    return hits / len(relevant)

In [11]:
def hit_rate_at_k(recommended, relevant, k):
    recommended = recommended[:k]

    return float(
        any(item in relevant for item in recommended)
    )

In [12]:
def ndcg_at_k(recommended, relevant, k):
    recommended = recommended[:k]

    dcg = 0.0

    for rank, item in enumerate(recommended, start=1):
        if item in relevant:
            dcg += 1 / np.log2(rank + 1)

    ideal_hits = min(len(relevant), k)

    if ideal_hits == 0:
        return 0.0

    idcg = sum(
        1 / np.log2(rank + 1)
        for rank in range(1, ideal_hits + 1)
    )

    return dcg / idcg

In [13]:
def evaluate_popularity(k=10):

    precision_scores = []
    recall_scores = []
    ndcg_scores = []
    hit_scores = []

    for user_id, relevant_items in ground_truth.items():

        recommendations = recommend_popular(
            user_id,
            k=k
        )

        precision_scores.append(
            precision_at_k(
                recommendations,
                relevant_items,
                k
            )
        )

        recall_scores.append(
            recall_at_k(
                recommendations,
                relevant_items,
                k
            )
        )

        ndcg_scores.append(
            ndcg_at_k(
                recommendations,
                relevant_items,
                k
            )
        )

        hit_scores.append(
            hit_rate_at_k(
                recommendations,
                relevant_items,
                k
            )
        )

    return {
        "Precision@K": np.mean(precision_scores),
        "Recall@K": np.mean(recall_scores),
        "NDCG@K": np.mean(ndcg_scores),
        "HitRate@K": np.mean(hit_scores)
    }

In [14]:
results_10 = evaluate_popularity(k=10)

print(results_10)

{'Precision@K': 0.0006169183903883433, 'Recall@K': 0.004860711451087305, 'NDCG@K': 0.002540538709161561, 'HitRate@K': 0.006058411264725306}


In [15]:
results = []

for k in [5, 10, 20]:

    metrics = evaluate_popularity(k=k)

    results.append({
        "Model": "Popularity",
        "K": k,
        **metrics
    })

results_df = pd.DataFrame(results)

print(results_df)

        Model   K  Precision@K  Recall@K    NDCG@K  HitRate@K
0  Popularity   5     0.000736  0.002944  0.001915   0.003677
1  Popularity  10     0.000617  0.004861  0.002541   0.006058
2  Popularity  20     0.000540  0.008072  0.003396   0.010523


In [17]:
results_df.to_csv(
    "../reports/model_results.csv",
    index=False
)

In [19]:
import pandas as pd

properties_1 = pd.read_csv(
    "../data/raw/item_properties_part1.csv"
)

print(properties_1.shape)
print(properties_1.head())
print(properties_1.info())

print("\nUnique properties:")
print(properties_1["property"].nunique())

print("\nTop properties:")
print(properties_1["property"].value_counts().head(20))

(10999999, 4)
       timestamp  itemid    property                            value
0  1435460400000  460429  categoryid                             1338
1  1441508400000  206783         888          1116713 960601 n277.200
2  1439089200000  395014         400  n552.000 639502 n720.000 424566
3  1431226800000   59481         790                       n15360.000
4  1431831600000  156781         917                           828513
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 10999999 entries, 0 to 10999998
Data columns (total 4 columns):
 #   Column     Dtype 
---  ------     ----- 
 0   timestamp  int64 
 1   itemid     int64 
 2   property   object
 3   value      object
dtypes: int64(2), object(2)
memory usage: 335.7+ MB
None

Unique properties:
1097

Top properties:
property
888           1629817
790            970800
available      817387
categoryid     426305
6              343207
283            323681
776            311654
678            261829
364            256340
202     

In [20]:
properties_2 = pd.read_csv(
    "../data/raw/item_properties_part2.csv"
)

print(properties_2.shape)
print(properties_2.head())
print(properties_2.info())

print("\nUnique properties:")
print(properties_2["property"].nunique())

print("\nTop properties:")
print(properties_2["property"].value_counts().head(20))

(9275903, 4)
       timestamp  itemid property            value
0  1433041200000  183478      561           769062
1  1439694000000  132256      976  n26.400 1135780
2  1435460400000  420307      921  1149317 1257525
3  1431831600000  403324      917          1204143
4  1435460400000  230701      521           769062
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 9275903 entries, 0 to 9275902
Data columns (total 4 columns):
 #   Column     Dtype 
---  ------     ----- 
 0   timestamp  int64 
 1   itemid     int64 
 2   property   object
 3   value      object
dtypes: int64(2), object(2)
memory usage: 283.1+ MB
None

Unique properties:
1094

Top properties:
property
888           1370581
790            819716
available      686252
categoryid     361909
6              288264
283            273738
776            262566
364            220146
678            220137
202            205954
112            190951
764            190811
917            190790
159            190551
839            

In [21]:
category_tree = pd.read_csv(
    "../data/raw/category_tree.csv"
)

print(category_tree.shape)
print(category_tree.head())
print(category_tree.info())

(1669, 2)
   categoryid  parentid
0        1016     213.0
1         809     169.0
2         570       9.0
3        1691     885.0
4         536    1691.0
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1669 entries, 0 to 1668
Data columns (total 2 columns):
 #   Column      Non-Null Count  Dtype  
---  ------      --------------  -----  
 0   categoryid  1669 non-null   int64  
 1   parentid    1644 non-null   float64
dtypes: float64(1), int64(1)
memory usage: 26.2 KB
None


In [22]:
print("PART 1")

print("Unique items:", properties_1["itemid"].nunique())

print(
    "Items with category:",
    properties_1.loc[
        properties_1["property"] == "categoryid",
        "itemid"
    ].nunique()
)

print(
    "Items with availability:",
    properties_1.loc[
        properties_1["property"] == "available",
        "itemid"
    ].nunique()
)

print(
    "Unique properties:",
    properties_1["property"].nunique()
)

PART 1
Unique items: 417053
Items with category: 229341
Items with availability: 236884
Unique properties: 1097


In [23]:
print("\nPART 2")

print("Unique items:", properties_2["itemid"].nunique())

print(
    "Items with category:",
    properties_2.loc[
        properties_2["property"] == "categoryid",
        "itemid"
    ].nunique()
)

print(
    "Items with availability:",
    properties_2.loc[
        properties_2["property"] == "available",
        "itemid"
    ].nunique()
)

print(
    "Unique properties:",
    properties_2["property"].nunique()
)


PART 2
Unique items: 417053
Items with category: 194993
Items with availability: 201374
Unique properties: 1094


In [24]:
import pandas as pd

interactions = pd.read_parquet(
    "../data/processed/interactions.parquet"
)

print(interactions.shape)
print(interactions.head())

(2756101, 5)
   user_id  item_id event               timestamp  interaction_strength
0   257597   355908  view 2015-06-02 05:02:12.117                     1
1   992329   248676  view 2015-06-02 05:50:14.164                     1
2   111016   318965  view 2015-06-02 05:13:19.827                     1
3   483717   253185  view 2015-06-02 05:12:35.914                     1
4   951259   367447  view 2015-06-02 05:02:17.106                     1


In [25]:
interaction_items = set(
    interactions["item_id"].unique()
)

property_items_1 = set(
    properties_1["itemid"].unique()
)

property_items_2 = set(
    properties_2["itemid"].unique()
)

property_items = property_items_1 | property_items_2

print("Interacted items:", len(interaction_items))
print("Items with metadata:", len(property_items))

overlap = interaction_items & property_items

print("Interacted items with metadata:", len(overlap))

print(
    "Metadata coverage:",
    len(overlap) / len(interaction_items) * 100
)

Interacted items: 235061
Items with metadata: 417053
Interacted items with metadata: 185246
Metadata coverage: 78.80762865809301


In [26]:
property_counts_1 = (
    properties_1
    .groupby("itemid")
    .size()
)

property_counts_2 = (
    properties_2
    .groupby("itemid")
    .size()
)

print("PART 1")
print(property_counts_1.describe())

print("\nPART 2")
print(property_counts_2.describe())

PART 1
count    417053.000000
mean         26.375542
std          17.241695
min           2.000000
25%          14.000000
50%          20.000000
75%          34.000000
max         322.000000
dtype: float64

PART 2
count    417053.000000
mean         22.241545
std          15.303822
min           1.000000
25%          12.000000
50%          16.000000
75%          29.000000
max         278.000000
dtype: float64


In [27]:
for name, df in [
    ("PART 1", properties_1),
    ("PART 2", properties_2)
]:
    print(f"\n{name}")

    print(
        df["property"]
        .value_counts()
        .loc[lambda x: x.index.isin(
            ["categoryid", "available"]
        )]
    )


PART 1
property
available     817387
categoryid    426305
Name: count, dtype: int64

PART 2
property
available     686252
categoryid    361909
Name: count, dtype: int64


In [28]:
train_items = set(
    train["item_id"].unique()
)

print("Training items:", len(train_items))

Training items: 200974


In [29]:
props1_train = properties_1[
    properties_1["itemid"].isin(train_items)
].copy()

props2_train = properties_2[
    properties_2["itemid"].isin(train_items)
].copy()

print("Part 1 filtered:", props1_train.shape)
print("Part 2 filtered:", props2_train.shape)

Part 1 filtered: (4877980, 4)
Part 2 filtered: (4106838, 4)


In [30]:
props1_train["feature"] = (
    props1_train["property"].astype(str)
    + "="
    + props1_train["value"].astype(str)
)

props2_train["feature"] = (
    props2_train["property"].astype(str)
    + "="
    + props2_train["value"].astype(str)
)

In [31]:
product_features = pd.concat(
    [
        props1_train[["itemid", "feature"]],
        props2_train[["itemid", "feature"]]
    ],
    ignore_index=True
)

print(product_features.shape)
print(product_features.head())

(8984818, 2)
   itemid                              feature
0  460429                      categoryid=1338
1  395014  400=n552.000 639502 n720.000 424566
2   59481                       790=n15360.000
3  156781                           917=828513
4  285026                          available=0


In [32]:
product_features = (
    product_features
    .drop_duplicates(["itemid", "feature"])
    .reset_index(drop=True)
)

print("After deduplication:", product_features.shape)

After deduplication: (5133338, 2)


In [33]:
print(
    "Unique products:",
    product_features["itemid"].nunique()
)

print(
    "Unique features:",
    product_features["feature"].nunique()
)

Unique products: 160670
Unique features: 1080096


In [34]:
import pandas as pd

# Keep the original product_features untouched

expanded_1 = props1_train[
    ["itemid", "property", "value"]
].copy()

expanded_2 = props2_train[
    ["itemid", "property", "value"]
].copy()

expanded = pd.concat(
    [expanded_1, expanded_2],
    ignore_index=True
)

print("Raw filtered rows:", len(expanded))

Raw filtered rows: 8984818


In [35]:
expanded["value"] = expanded["value"].astype(str)

expanded["value_tokens"] = expanded["value"].str.split()

In [36]:
expanded = expanded.explode(
    "value_tokens",
    ignore_index=True
)

print("Rows after value expansion:", len(expanded))

Rows after value expansion: 23031079


In [37]:
expanded["feature"] = (
    expanded["property"].astype(str)
    + "="
    + expanded["value_tokens"].astype(str)
)

In [38]:
expanded = expanded[
    ["itemid", "feature"]
].drop_duplicates()

In [39]:
print("Unique products:", expanded["itemid"].nunique())

print(
    "Unique features:",
    expanded["feature"].nunique()
)

print(
    "Total item-feature pairs:",
    len(expanded)
)

Unique products: 160670
Unique features: 1027303
Total item-feature pairs: 10261448


In [40]:
feature_frequency = (
    expanded
    .groupby("feature")["itemid"]
    .nunique()
)

In [41]:
print(feature_frequency.describe())

count    1.027303e+06
mean     9.988726e+00
std      3.775623e+02
min      1.000000e+00
25%      1.000000e+00
50%      1.000000e+00
75%      1.000000e+00
max      1.606700e+05
Name: itemid, dtype: float64


In [42]:
print(
    "Features appearing once:",
    (feature_frequency == 1).sum()
)

print(
    "Features appearing <=5 times:",
    (feature_frequency <= 5).sum()
)

print(
    "Features appearing >=5 times:",
    (feature_frequency >= 5).sum()
)

Features appearing once: 830034
Features appearing <=5 times: 951721
Features appearing >=5 times: 86490


In [48]:
MIN_FEATURE_FREQUENCY = 5

In [47]:
valid_features = feature_frequency[
    feature_frequency >= MIN_FEATURE_FREQUENCY
].index

In [46]:
expanded_filtered = expanded[
    expanded["feature"].isin(valid_features)
].copy()

In [49]:
print(
    "Products:",
    expanded_filtered["itemid"].nunique()
)

print(
    "Features:",
    expanded_filtered["feature"].nunique()
)

print(
    "Item-feature pairs:",
    len(expanded_filtered)
)

Products: 160670
Features: 86490
Item-feature pairs: 9148457


In [ ]:
product_documents = (
    expanded_filtered
    .groupby("itemid")["feature"]
    .apply(lambda x: " ".join(x))
)

print("Products:", len(product_documents))
print(product_documents.head())

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer

In [ ]:
vectorizer = TfidfVectorizer(
    token_pattern=r"(?u)\S+",
    min_df=5,
    dtype=np.float32
)

In [ ]:
product_matrix = vectorizer.fit_transform(
    product_documents.values
)

print("Matrix shape:", product_matrix.shape)
print("Non-zero values:", product_matrix.nnz)

In [ ]:
total_elements = (
    product_matrix.shape[0] *
    product_matrix.shape[1]
)

sparsity = 1 - (
    product_matrix.nnz /
    total_elements
)

print("Sparsity:", sparsity)
print("Density:", 1 - sparsity)

In [ ]:
product_ids = product_documents.index.to_numpy()

In [ ]:
import joblib

joblib.dump(
    vectorizer,
    "../models/content_vectorizer.joblib"
)

np.save(
    "../models/product_ids.npy",
    product_ids
)

from scipy.sparse import save_npz

save_npz(
    "../models/product_tfidf.npz",
    product_matrix
)